# 一步一步学会 BPE

这个 notebook 用最小可运行代码带你理解 Byte Pair Encoding，也就是 GPT 类模型常用 tokenizer 的核心思想。

学习路线：

1. 先理解文本、Unicode、UTF-8 bytes 的关系。
2. 用一个玩具语料手算 BPE 的合并过程。
3. 写出训练 BPE merges 的代码。
4. 写出 encode / decode。
5. 理解 pre-tokenization 和 special tokens 在 CS336 作业里的位置。


## 0. 先建立直觉

BPE 的核心非常朴素：

> 从最小单位开始，反复找到最常一起出现的相邻 token 对，把它们合并成一个新 token。

如果语料里经常出现 `l` 后面跟 `o`，我们可以把 `(l, o)` 合并成 `lo`。如果 `lo` 后面经常跟 `w`，再合并成 `low`。这样常见片段会变成短 token，罕见片段仍然可以拆回更小单位。

In [1]:
from collections import Counter, defaultdict
from pprint import pprint


## 1. 文本不是模型真正看到的东西

Python 字符串是 Unicode 文本；真实文件通常会被编码成 bytes。CS336 的 BPE 作业采用 byte-level BPE：初始词表就是 256 个单字节 token。

这带来一个重要好处：任何 UTF-8 文本都能被表示，不会出现未知字符。

In [4]:
examples = ["hello", "你好", "🙂"]

for text in examples:
    raw = text.encode("utf-8")
    print(repr(text), "->", list(raw), "->", raw)


'hello' -> [104, 101, 108, 108, 111] -> b'hello'
'你好' -> [228, 189, 160, 229, 165, 189] -> b'\xe4\xbd\xa0\xe5\xa5\xbd'
'🙂' -> [240, 159, 153, 130] -> b'\xf0\x9f\x99\x82'


## 2. 用字符级例子手算一次 BPE

先不用 bytes，先用字符做一个玩具版本，更容易看懂。假设语料里有这些词和频次：

In [9]:
word_counts = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
}

def split_word_to_chars(word):
    return tuple(word)
splits = {word: split_word_to_chars(word) for word in word_counts}
pprint(splits)



{'low': ('l', 'o', 'w'),
 'lower': ('l', 'o', 'w', 'e', 'r'),
 'newest': ('n', 'e', 'w', 'e', 's', 't'),
 'widest': ('w', 'i', 'd', 'e', 's', 't')}


## 3. 第一步：统计相邻 pair

BPE 每一轮都问同一个问题：当前所有 token 序列里，哪个相邻 token pair 的加权出现次数最高？

In [7]:
def count_adjacent_pairs(splits, word_counts):
    pair_counts = Counter()
    for word, tokens in splits.items():
        count = word_counts[word]
        for pair in zip(tokens, tokens[1:]):
            pair_counts[pair] += count
    return pair_counts

pair_counts = count_adjacent_pairs(splits, word_counts)
pair_counts.most_common(10)


[(('e', 's'), 9),
 (('s', 't'), 9),
 (('w', 'e'), 8),
 (('l', 'o'), 7),
 (('o', 'w'), 7),
 (('n', 'e'), 6),
 (('e', 'w'), 6),
 (('w', 'i'), 3),
 (('i', 'd'), 3),
 (('d', 'e'), 3)]

## 4. 第二步：把最高频 pair 合并

如果最高频 pair 是 `('e', 's')`，就把所有相邻的 `e s` 替换成 `es`。注意只合并相邻位置。

In [10]:
def merge_pair_in_tokens(tokens, pair, new_token):
    merged = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merged.append(new_token)
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return tuple(merged)

best_pair, best_count = pair_counts.most_common(1)[0]
new_token = "".join(best_pair)
print("best pair:", best_pair, "count:", best_count, "new token:", new_token)

splits_after_one_merge = {
    word: merge_pair_in_tokens(tokens, best_pair, new_token)
    for word, tokens in splits.items()
}
pprint(splits_after_one_merge)


best pair: ('e', 's') count: 9 new token: es
{'low': ('l', 'o', 'w'),
 'lower': ('l', 'o', 'w', 'e', 'r'),
 'newest': ('n', 'e', 'w', 'es', 't'),
 'widest': ('w', 'i', 'd', 'es', 't')}


## 5. 把一轮合并变成训练循环

训练 BPE merges 就是重复上面两步：统计 pair，选择最高频 pair，合并，记录 merge 顺序。这个顺序很重要，之后 encode 新文本时也要按同样顺序应用。

In [14]:
def train_char_bpe(word_counts, num_merges):
    splits = {word: tuple(word) for word in word_counts}
    merges = []

    for step in range(num_merges):
        pair_counts = count_adjacent_pairs(splits, word_counts)
        if not pair_counts:
            break

        best_pair, best_count = pair_counts.most_common(1)[0]
        new_token = "".join(best_pair)
        merges.append((best_pair, new_token, best_count))

        splits = {
            word: merge_pair_in_tokens(tokens, best_pair, new_token)
            for word, tokens in splits.items()
        }

        print(f"step {step + 1}: merge {best_pair} -> {new_token!r}, count={best_count}")
        pprint(splits)
        print()

    return merges, splits

char_merges, final_char_splits = train_char_bpe(word_counts, num_merges=8)


step 1: merge ('e', 's') -> 'es', count=9
{'low': ('l', 'o', 'w'),
 'lower': ('l', 'o', 'w', 'e', 'r'),
 'newest': ('n', 'e', 'w', 'es', 't'),
 'widest': ('w', 'i', 'd', 'es', 't')}

step 2: merge ('es', 't') -> 'est', count=9
{'low': ('l', 'o', 'w'),
 'lower': ('l', 'o', 'w', 'e', 'r'),
 'newest': ('n', 'e', 'w', 'est'),
 'widest': ('w', 'i', 'd', 'est')}

step 3: merge ('l', 'o') -> 'lo', count=7
{'low': ('lo', 'w'),
 'lower': ('lo', 'w', 'e', 'r'),
 'newest': ('n', 'e', 'w', 'est'),
 'widest': ('w', 'i', 'd', 'est')}

step 4: merge ('lo', 'w') -> 'low', count=7
{'low': ('low',),
 'lower': ('low', 'e', 'r'),
 'newest': ('n', 'e', 'w', 'est'),
 'widest': ('w', 'i', 'd', 'est')}

step 5: merge ('n', 'e') -> 'ne', count=6
{'low': ('low',),
 'lower': ('low', 'e', 'r'),
 'newest': ('ne', 'w', 'est'),
 'widest': ('w', 'i', 'd', 'est')}

step 6: merge ('ne', 'w') -> 'new', count=6
{'low': ('low',),
 'lower': ('low', 'e', 'r'),
 'newest': ('new', 'est'),
 'widest': ('w', 'i', 'd', 'est')}

s

## 6. 进入 byte-level BPE

CS336 作业里更接近真实 GPT tokenizer 的版本：

- 初始 token 是单个 byte，例如 `b'a'` 或 `b'\xe4'`。
- 每次 merge 产生一个更长的 bytes token，例如 `b't' + b'h' -> b'th'`。
- 词表 `vocab` 通常是 `id -> bytes`。
- merges 通常是有顺序的 `(bytes, bytes)` 列表。

In [58]:
def bytes_to_initial_tokens(text):
    return tuple(bytes([b]) for b in text.encode("utf-8"))

bytes_to_initial_tokens("hello 你好")


(b'h',
 b'e',
 b'l',
 b'l',
 b'o',
 b' ',
 b'\xe4',
 b'\xbd',
 b'\xa0',
 b'\xe5',
 b'\xa5',
 b'\xbd')

## 7. 为什么需要 pre-tokenization

如果直接在整篇文章上做 BPE，merge 可能跨过空格、标点、特殊 token 边界，训练和编码都会更难控制。

实际 tokenizer 通常先用正则把文本切成 pre-tokens。BPE 只在每个 pre-token 内部合并。下面先用一个简化版：按非空白片段切分，同时保留空白。

In [57]:
import re

def simple_pretokenize(text):
    return re.findall(r"\S+|\s+", text)

simple_pretokenize("hello  world!\n你好")




['hello', '  ', 'world!', '\n', '你好']

## 8. 写一个最小 byte-level BPE trainer

这个版本为了学习而写得直观，不追求速度。你之后做作业时，可以把这里的逻辑优化成更高效的数据结构。

In [59]:
def make_pretoken_counts(corpus):
    counts = Counter()
    for text in corpus:
        for pretoken in simple_pretokenize(text):
            counts[pretoken.encode("utf-8")] += 1
    return counts

def initial_byte_splits(pretoken_counts):
    return {
        pretoken: tuple(bytes([b]) for b in pretoken)
        for pretoken in pretoken_counts
    }

def count_byte_pairs(splits, pretoken_counts):
    pair_counts = Counter()
    for pretoken, tokens in splits.items():
        count = pretoken_counts[pretoken]
        for pair in zip(tokens, tokens[1:]):
            pair_counts[pair] += count
    return pair_counts

def merge_byte_pair(tokens, pair):
    new_token = pair[0] + pair[1]
    merged = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merged.append(new_token)
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return tuple(merged)

def train_byte_bpe(corpus, vocab_size):
    assert vocab_size >= 256

    vocab = {i: bytes([i]) for i in range(256)}
    merges = []
    pretoken_counts = make_pretoken_counts(corpus)
    splits = initial_byte_splits(pretoken_counts)
    print(pretoken_counts)
    while len(vocab) < vocab_size:
        pair_counts = count_byte_pairs(splits, pretoken_counts)
        if not pair_counts:
            break

        best_pair, best_count = pair_counts.most_common(1)[0]
        new_token = best_pair[0] + best_pair[1]
        new_id = len(vocab)

        merges.append(best_pair)
        vocab[new_id] = new_token

        splits = {
            pretoken: merge_byte_pair(tokens, best_pair)
            for pretoken, tokens in splits.items()
        }

        if len(vocab) <= 270:
            print(f"id={new_id}: merge {best_pair} -> {new_token!r}, count={best_count}")

    return vocab, merges


In [60]:
tiny_corpus = [
    "low lower lowest",
    "new newer newest",
    "the theater is there",
    "你好 你好 hello",
]

vocab, merges = train_byte_bpe(tiny_corpus, vocab_size=280)
print("num vocab:", len(vocab))
print("num merges:", len(merges))


Counter({b' ': 9, b'\xe4\xbd\xa0\xe5\xa5\xbd': 2, b'low': 1, b'lower': 1, b'lowest': 1, b'new': 1, b'newer': 1, b'newest': 1, b'the': 1, b'theater': 1, b'is': 1, b'there': 1, b'hello': 1})
id=256: merge (b'l', b'o') -> b'lo', count=4
id=257: merge (b'w', b'e') -> b'we', count=4
id=258: merge (b'h', b'e') -> b'he', count=4
id=259: merge (b'n', b'e') -> b'ne', count=3
id=260: merge (b't', b'he') -> b'the', count=3
id=261: merge (b'lo', b'we') -> b'lowe', count=2
id=262: merge (b's', b't') -> b'st', count=2
id=263: merge (b'ne', b'we') -> b'newe', count=2
id=264: merge (b'\xe4', b'\xbd') -> b'\xe4\xbd', count=2
id=265: merge (b'\xe4\xbd', b'\xa0') -> b'\xe4\xbd\xa0', count=2
id=266: merge (b'\xe4\xbd\xa0', b'\xe5') -> b'\xe4\xbd\xa0\xe5', count=2
id=267: merge (b'\xe4\xbd\xa0\xe5', b'\xa5') -> b'\xe4\xbd\xa0\xe5\xa5', count=2
id=268: merge (b'\xe4\xbd\xa0\xe5\xa5', b'\xbd') -> b'\xe4\xbd\xa0\xe5\xa5\xbd', count=2
id=269: merge (b'lo', b'w') -> b'low', count=1
num vocab: 280
num merges: 24

## 9. 看看学到了什么 token

词表里前 256 个 token 是单 byte；后面的 token 才是通过 merge 学出来的多 byte token。

In [40]:
for token_id in range(256, len(vocab)):
    token = vocab[token_id]
    readable = token.decode("utf-8", errors="replace")
    print(token_id, token, repr(readable))


256 b'lo' 'lo'
257 b'we' 'we'
258 b'he' 'he'
259 b'ne' 'ne'
260 b'the' 'the'
261 b'lowe' 'lowe'
262 b'st' 'st'
263 b'newe' 'newe'
264 b'\xe4\xbd' '�'
265 b'\xe4\xbd\xa0' '你'
266 b'\xe4\xbd\xa0\xe5' '你�'
267 b'\xe4\xbd\xa0\xe5\xa5' '你�'
268 b'\xe4\xbd\xa0\xe5\xa5\xbd' '你好'
269 b'low' 'low'
270 b'lower' 'lower'
271 b'lowest' 'lowest'
272 b'new' 'new'
273 b'newer' 'newer'
274 b'newest' 'newest'
275 b'thea' 'thea'
276 b'theat' 'theat'
277 b'theate' 'theate'
278 b'theater' 'theater'
279 b'is' 'is'


## 10. Encode：对新文本应用已学到的 merges

训练时记录了 merge 顺序。编码时：

1. 把文本 pre-tokenize。
2. 每个 pre-token 先变成单 byte tokens。
3. 按训练顺序依次尝试 merge。
4. 把最终 bytes token 查到 token id。

In [61]:
def build_token_to_id(vocab):
    return {token: token_id for token_id, token in vocab.items()}

def apply_merges_to_pretoken(pretoken_bytes, merges):
    tokens = tuple(bytes([b]) for b in pretoken_bytes)
    for pair in merges:
        tokens = merge_byte_pair(tokens, pair)
    return tokens

def encode(text, vocab, merges):
    token_to_id = build_token_to_id(vocab)
    # print(token_to_id)
    ids = []
    pieces = []

    for pretoken in simple_pretokenize(text):
        pretoken_bytes = pretoken.encode("utf-8")
        tokens = apply_merges_to_pretoken(pretoken_bytes, merges)
        pieces.extend(tokens)
        ids.extend(token_to_id[token] for token in tokens)

    return ids, pieces
ids, pieces = encode("hello there 你好", vocab, merges)
print(ids)
print(pieces)


{b'\x00': 0, b'\x01': 1, b'\x02': 2, b'\x03': 3, b'\x04': 4, b'\x05': 5, b'\x06': 6, b'\x07': 7, b'\x08': 8, b'\t': 9, b'\n': 10, b'\x0b': 11, b'\x0c': 12, b'\r': 13, b'\x0e': 14, b'\x0f': 15, b'\x10': 16, b'\x11': 17, b'\x12': 18, b'\x13': 19, b'\x14': 20, b'\x15': 21, b'\x16': 22, b'\x17': 23, b'\x18': 24, b'\x19': 25, b'\x1a': 26, b'\x1b': 27, b'\x1c': 28, b'\x1d': 29, b'\x1e': 30, b'\x1f': 31, b' ': 32, b'!': 33, b'"': 34, b'#': 35, b'$': 36, b'%': 37, b'&': 38, b"'": 39, b'(': 40, b')': 41, b'*': 42, b'+': 43, b',': 44, b'-': 45, b'.': 46, b'/': 47, b'0': 48, b'1': 49, b'2': 50, b'3': 51, b'4': 52, b'5': 53, b'6': 54, b'7': 55, b'8': 56, b'9': 57, b':': 58, b';': 59, b'<': 60, b'=': 61, b'>': 62, b'?': 63, b'@': 64, b'A': 65, b'B': 66, b'C': 67, b'D': 68, b'E': 69, b'F': 70, b'G': 71, b'H': 72, b'I': 73, b'J': 74, b'K': 75, b'L': 76, b'M': 77, b'N': 78, b'O': 79, b'P': 80, b'Q': 81, b'R': 82, b'S': 83, b'T': 84, b'U': 85, b'V': 86, b'W': 87, b'X': 88, b'Y': 89, b'Z': 90, b'[': 91,

## 11. Decode：把 token ids 拼回 bytes，再 UTF-8 解码

byte-level BPE 的 decode 很优雅：查表得到 bytes，全部拼起来，然后 `.decode('utf-8')`。

In [48]:
def decode(ids, vocab):
    raw = b"".join(vocab[token_id] for token_id in ids)
    return raw.decode("utf-8", errors="replace")

decoded = decode(ids, vocab)
decoded


'hello there 你好'

In [49]:
text = "hello there 你好"
ids, pieces = encode(text, vocab, merges)
assert decode(ids, vocab) == text
print("round trip ok:", ids)


round trip ok: [258, 108, 256, 32, 260, 114, 101, 32, 268]


## 12. Special tokens 为什么要特殊处理

像 `<|endoftext|>` 这种 token 不应该被普通 BPE 拆成 `<`, `|`, `end`, ...。它通常被直接加入词表，并在 encode 时优先识别。

CS336 作业常见要求是：训练时按 special token 分割，避免 merge 跨过 special token；编码时如果允许 special token，就把它整体映射成一个 id。

In [50]:
special_token = "<|endoftext|>"
special_id = len(vocab)
vocab_with_special = dict(vocab)
vocab_with_special[special_id] = special_token.encode("utf-8")

def encode_with_one_special(text, vocab, merges, special_token):
    token_to_id = build_token_to_id(vocab)
    special_bytes = special_token.encode("utf-8")
    ids = []

    parts = text.split(special_token)
    for i, part in enumerate(parts):
        if part:
            part_ids, _ = encode(part, vocab, merges)
            ids.extend(part_ids)
        if i < len(parts) - 1:
            ids.append(token_to_id[special_bytes])
    return ids

text = "hello<|endoftext|>你好"
ids = encode_with_one_special(text, vocab_with_special, merges, special_token)
print(ids)
print(decode(ids, vocab_with_special))


[258, 108, 256, 280, 268]
hello<|endoftext|>你好


## 13. 对应到 CS336 assignment1-basics

你真正实现作业时，通常会把上面的教学版拆成这些部分：

- `train_bpe(input_path, vocab_size, special_tokens)`：读文件，pre-tokenize，统计 pair，训练 `vocab` 和 `merges`。
- `Tokenizer.encode(text)`：处理 special tokens，pre-tokenize，应用 merges，返回 ids。
- `Tokenizer.decode(ids)`：id 查 bytes，拼接，UTF-8 decode。

性能优化方向：

- 不要每轮从零扫描全部语料，作业大数据会很慢。
- 可以先统计 pre-token 频次，只对不同 pre-token 存 token 序列。
- pair 频次可以用堆或增量更新优化。
- 大文件可以用 `pretokenization_example.py` 里的 chunk boundary 思路并行预分词。

## 14. 小练习

你可以按顺序做这几个练习，基本就能把 BPE 吃透：

1. 修改 `tiny_corpus`，加入更多中文、英文和标点，观察学出来的 token。
2. 把 `vocab_size` 从 270 改到 300，看编码后的 token 数量是否减少。
3. 在 `train_byte_bpe` 里打印每轮 `pair_counts.most_common(5)`，观察竞争关系。
4. 手写一个更像 GPT-2 的 pre-tokenizer，然后替换 `simple_pretokenize`。
5. 思考：为什么 encode 必须按训练时的 merge 顺序，而不是每次重新找当前最高频 pair？

## 15. 一句话总结

BPE 不是在“理解语言”，它是在语料中寻找常见的相邻 byte/token 片段，并把这些片段压缩成更大的 token；因为从 bytes 开始，所以它永远能表示任意文本。